# 📚 Smart Library — Time Travel & Gouvernance
## Notebook 05 : Time Travel, Changelog et Audit des données (Apache Iceberg)

---

### Objectif
Démontrer les fonctionnalités avancées de **traçabilité et gouvernance** offertes par Iceberg.

### Fonctionnalités démontrées
| Fonctionnalité | Description | Cas d'usage |
|---|---|---|
| **Time Travel** | Lire les données à un `snapshot_id` passé | Audit financier, débogage |
| **Historique des snapshots** | `{table}.snapshots` / `{table}.history` | Traçabilité réglementaire |
| **Comparaison V1/V3** | Différences entre deux snapshots | Contrôle qualité |
| **Changelog** | Lire uniquement les changements (`create_changelog_view`) | Pipeline efficace |
| **Audit trail** | Qui a modifié quoi et quand | Conformité RGPD |

> 💡 Ces fonctionnalités sont le cœur de la valeur académique du projet :
> elles démontrent que le Lakehouse va bien au-delà du simple stockage.
>
> **Différence clé avec Hudi** : Hudi référence un état passé par un `_hoodie_commit_time`
> (chaîne de caractères, présente comme colonne technique dans chaque ligne). Iceberg
> référence un état passé par un `snapshot_id` (entier), consultable exclusivement via
> les *metadata tables* du catalogue — jamais mélangé aux données elles-mêmes.

---
## Étape 0 — Initialisation

In [1]:
import os, sys, time, re
from datetime import datetime
from pathlib import Path

# ── Tuer toute session existante ─────────────────────────────
try:
    from pyspark.sql import SparkSession
    SparkSession.getActiveSession().stop()
except:
    pass

# ── Java ──────────────────────────────────────────────────────
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PATH"]      = os.environ["JAVA_HOME"] + "/bin:" + os.environ.get("PATH", "")

# ── Jars Iceberg + MinIO ──────────────────────────────────────
os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--packages "
    "org.apache.iceberg:iceberg-spark-runtime-3.4_2.12:1.5.2,"
    "org.apache.iceberg:iceberg-aws-bundle:1.5.2,"
    "org.apache.hadoop:hadoop-aws:3.3.4,"
    "com.amazonaws:aws-java-sdk-bundle:1.12.262 "
    "pyspark-shell"
)

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

# ── Connexion MinIO + Lakekeeper ──────────────────────────────
MINIO_ENDPOINT       = "http://minio:9000"
MINIO_ACCESS         = "minioadmin"
MINIO_SECRET         = "minioadmin123"
CATALOG_NAME         = "lakehouse"
LAKEKEEPER_URI       = "http://lakekeeper:8181/catalog"
LAKEKEEPER_WAREHOUSE = "lakehouse"

NS_BRONZE = f"{CATALOG_NAME}.bronze"
NS_SILVER = f"{CATALOG_NAME}.silver"
NS_GOLD   = f"{CATALOG_NAME}.gold"

# ── Fonctions utilitaires Iceberg (équivalent hudi_options) ───
#
#  Avec Hudi : le type de table (COW/MOR) et la clé d'upsert se
#  déclarent au niveau de l'ÉCRITURE (options du DataFrameWriter).
#  Avec Iceberg : ce sont des PROPRIÉTÉS DE TABLE, et l'UPSERT
#  s'exprime explicitement en SQL via MERGE INTO.
#
def iceberg_table_properties(type_table="copy-on-write"):
    """
    type_table :
      - "copy-on-write" → équivalent Hudi COW
      - "merge-on-read"  → équivalent Hudi MOR
    """
    return {
        "format-version"    : "2",
        "write.delete.mode" : type_table,
        "write.update.mode" : type_table,
        "write.merge.mode"  : type_table,
    }

def create_or_replace_table(df, table_name, partition_cols=None, type_table="copy-on-write"):
    """Équivalent de : df.write.format('hudi')....mode('overwrite').save(chemin)"""
    # Iceberg (ClusteredWriter) exige que les lignes soient triées/regroupées
    # par colonne de partition AVANT l'écriture — contrairement à Hudi qui
    # gère le clustering en interne. Sans ce tri : "Incoming records violate
    # the writer assumption that records are clustered by spec and partition".
    if partition_cols:
        df = df.sort(*partition_cols)
    writer = df.writeTo(table_name)
    if partition_cols:
        writer = writer.partitionedBy(*[F.col(c) for c in partition_cols])
    for k, v in iceberg_table_properties(type_table).items():
        writer = writer.tableProperty(k, v)
    writer.createOrReplace()

def append_table(df, table_name):
    """Ajout de lignes (insert simple, pas d'upsert)."""
    df.writeTo(table_name).append()

def upsert_table(spark, df, table_name, key_cols):
    """UPSERT via MERGE INTO — équivalent fonctionnel de l'UPSERT automatique Hudi."""
    vue = f"_staging_{table_name.split('.')[-1]}"
    df.createOrReplaceTempView(vue)
    condition = " AND ".join([f"t.{c} = s.{c}" for c in key_cols])
    spark.sql(f"""
        MERGE INTO {table_name} t
        USING {vue} s
        ON {condition}
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)

def snapshots(spark, table_name):
    """Retourne l'historique des snapshots Iceberg (équivalent de la timeline Hudi)."""
    return spark.sql(f"SELECT * FROM {table_name}.snapshots ORDER BY committed_at")

# ── SparkSession ──────────────────────────────────────────────
spark = (
    SparkSession.builder
    .appName("SmartLibrary-TimeTravel")
    .master("local[2]")
    .config("spark.driver.memory", "2g")
    .config("spark.sql.extensions",
            "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config(f"spark.sql.catalog.{CATALOG_NAME}", "org.apache.iceberg.spark.SparkCatalog")
    .config(f"spark.sql.catalog.{CATALOG_NAME}.type", "rest")
    .config(f"spark.sql.catalog.{CATALOG_NAME}.uri", LAKEKEEPER_URI)
    .config(f"spark.sql.catalog.{CATALOG_NAME}.warehouse", LAKEKEEPER_WAREHOUSE)
    .config(f"spark.sql.catalog.{CATALOG_NAME}.io-impl",
            "org.apache.iceberg.aws.s3.S3FileIO")
    .config(f"spark.sql.catalog.{CATALOG_NAME}.s3.endpoint",           MINIO_ENDPOINT)
    .config(f"spark.sql.catalog.{CATALOG_NAME}.s3.access-key-id",     MINIO_ACCESS)
    .config(f"spark.sql.catalog.{CATALOG_NAME}.s3.secret-access-key", MINIO_SECRET)
    .config(f"spark.sql.catalog.{CATALOG_NAME}.s3.path-style-access", "true")
    .config("spark.sql.defaultCatalog", CATALOG_NAME)
    .config("spark.serializer",
            "org.apache.spark.serializer.KryoSerializer")
    .config("spark.hadoop.fs.s3a.endpoint",          MINIO_ENDPOINT)
    .config("spark.hadoop.fs.s3a.access.key",        MINIO_ACCESS)
    .config("spark.hadoop.fs.s3a.secret.key",        MINIO_SECRET)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl",
            "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

# ── Création des namespaces Medallion (idempotent) ────────────
for ns in (NS_BRONZE, NS_SILVER, NS_GOLD):
    spark.sql(f"CREATE NAMESPACE IF NOT EXISTS {ns}")

print(f"✅ SparkSession démarrée — version {spark.version}")
print(f"   Catalogue Iceberg : {CATALOG_NAME} (REST → Lakekeeper → Postgres)")
print(f"   Stockage          : {MINIO_ENDPOINT} (bucket lakehouse)")

SILVER_STOCK    = f"{NS_SILVER}.stock"
SILVER_LIVRES   = f"{NS_SILVER}.livres"
SILVER_COMMANDES= f"{NS_SILVER}.commandes"
SILVER_UTILISATEURS = f"{NS_SILVER}.utilisateurs"
GOLD_HISTORIQUE = f"{NS_GOLD}.historique_ventes"
GOLD_TOP_LIVRES = f"{NS_GOLD}.top_livres"
GOLD_TENDANCES_RT = f"{NS_GOLD}.tendances_temps_reel"

✅ SparkSession démarrée — version 3.4.0
   Catalogue Iceberg : lakehouse (REST → Lakekeeper → Postgres)
   Stockage          : http://minio:9000 (bucket lakehouse)


---
## Étape 1 — Historique des snapshots Iceberg

> 💡 Iceberg maintient un fichier de métadonnées (`metadata.json`) qui référence tous les
> snapshots successifs d'une table — c'est l'équivalent du dossier `.hoodie/` de Hudi,
> mais exposé nativement via les *metadata tables* SQL (`{table}.snapshots`, `{table}.history`).

In [2]:
def afficher_timeline(table_name, nom_affiche):
    """Affiche l'historique des snapshots d'une table Iceberg."""
    snaps = spark.sql(f"""
        SELECT snapshot_id, committed_at, operation, summary['total-records'] AS total_records
        FROM {table_name}.snapshots
        ORDER BY committed_at
    """).collect()

    print(f"\n  Table : {nom_affiche}")
    print(f"  Nombre de snapshots : {len(snaps)}")
    print(f"  {'#':<4} {'Snapshot ID':<22} {'Opération':<12} {'Lignes totales':>15}")
    print(f"  {'-'*4} {'-'*22} {'-'*12} {'-'*15}")

    for i, row in enumerate(snaps):
        print(f"  {i+1:<4} {str(row['snapshot_id']):<22} {row['operation']:<12} {str(row['total_records']):>15}")

    return [r["snapshot_id"] for r in snaps]

print("=" * 55)
print("  TIMELINE ICEBERG — HISTORIQUE DES SNAPSHOTS")
print("=" * 55)

snapshots_stock = afficher_timeline(SILVER_STOCK,     "silver.stock")
snapshots_hist  = afficher_timeline(GOLD_HISTORIQUE,  "gold.historique_ventes")

  TIMELINE ICEBERG — HISTORIQUE DES SNAPSHOTS

  Table : silver.stock
  Nombre de snapshots : 2
  #    Snapshot ID            Opération     Lignes totales
  ---- ---------------------- ------------ ---------------
  1    1565405350769361658    append                   500
  2    1537247837498438725    overwrite                500

  Table : gold.historique_ventes
  Nombre de snapshots : 2
  #    Snapshot ID            Opération     Lignes totales
  ---- ---------------------- ------------ ---------------
  1    3731074348899197005    append                  4003
  2    3362267105965717146    overwrite               4003


---
## Étape 2 — Scénario Time Travel complet
### Simulation d'une erreur de prix → correction → audit

In [3]:
# ── État V1 : snapshot initial du stock ───────────────────────
print("=" * 62)
print("  SCÉNARIO TIME TRAVEL")
print("  Erreur : prix mal appliqués → correction → audit")
print("=" * 62)

df_stock_actuel = spark.table(SILVER_STOCK)

# Sauvegarder snapshot V1
snapshot_v1 = spark.sql(f"SELECT snapshot_id FROM {SILVER_STOCK}.snapshots ORDER BY committed_at DESC LIMIT 1").first()["snapshot_id"]
n_v1 = df_stock_actuel.count()

valeur_v1 = (
    df_stock_actuel
    .agg(F.round(F.sum("valeur_stock"), 2).alias("total"))
    .first()["total"]
)

print(f"\n  V1 — Snapshot  : {snapshot_v1}")
print(f"  V1 — Lignes    : {n_v1:,}")
print(f"  V1 — Valeur stock totale : {valeur_v1:,} €")
print()

# Aperçu V1
df_stock_actuel.select(
    "stock_id", "titre", "quantite_disponible", "valeur_stock", "statut_stock"
).orderBy(F.col("valeur_stock").desc()).show(5, truncate=True)

  SCÉNARIO TIME TRAVEL
  Erreur : prix mal appliqués → correction → audit

  V1 — Snapshot  : 1537247837498438725
  V1 — Lignes    : 500
  V1 — Valeur stock totale : 1,586,882.99 €

+----------+--------------------+-------------------+------------+------------+
|  stock_id|               titre|quantite_disponible|valeur_stock|statut_stock|
+----------+--------------------+-------------------+------------+------------+
|STK-BK0292|Répéter Détail No...|                194|      9670.9|   Suffisant|
|STK-BK0390|Vision Souhaiter ...|                192|      9446.4|   Suffisant|
|STK-BK0301|       Réel Vêtement|                186|     9007.98|   Suffisant|
|STK-BK0275|Jeune Imposer Coe...|                189|     8986.95|   Suffisant|
|STK-BK0396|Écouter Importanc...|                187|     8946.08|   Suffisant|
+----------+--------------------+-------------------+------------+------------+
only showing top 5 rows



In [4]:
# ── Simuler une erreur : majoration de 50% sur 30 articles ───
print("  Simulation d'une erreur : majoration accidentelle +50%")
print("-" * 62)

df_erreur = (
    spark.table(SILVER_STOCK)
    .limit(30)
    .withColumn("valeur_stock",
                F.round(F.col("valeur_stock") * 1.5, 2))  # Erreur : +50%
    .withColumn("updated_at",
                F.lit(datetime.now().isoformat()).cast(StringType()))
)

upsert_table(spark, df_erreur, SILVER_STOCK, key_cols=["stock_id"])

# État V2 — avec erreur
df_v2 = spark.table(SILVER_STOCK)
snapshot_v2 = spark.sql(f"SELECT snapshot_id FROM {SILVER_STOCK}.snapshots ORDER BY committed_at DESC LIMIT 1").first()["snapshot_id"]
valeur_v2 = (
    df_v2
    .agg(F.round(F.sum("valeur_stock"), 2).alias("total"))
    .first()["total"]
)

print(f"  V2 — Snapshot  : {snapshot_v2}")
print(f"  V2 — Valeur stock (avec erreur) : {valeur_v2:,} €")
print(f"  Écart dû à l'erreur             : +{round(valeur_v2 - valeur_v1, 2):,} €")

  Simulation d'une erreur : majoration accidentelle +50%
--------------------------------------------------------------
  V2 — Snapshot  : 6695272034873840362
  V2 — Valeur stock (avec erreur) : 1,701,390.18 €
  Écart dû à l'erreur             : +114,507.19 €


In [5]:
# ── TIME TRAVEL : relire l'état V1 pour corriger ──────────────
print("\n  TIME TRAVEL → retour à l'état V1")
print("-" * 62)

# Relire l'état AVANT l'erreur
df_stock_v1 = (
    spark.read.format("iceberg")
    .option("snapshot-id", snapshot_v1)
    .load(SILVER_STOCK)
)

valeur_relu_v1 = (
    df_stock_v1
    .agg(F.round(F.sum("valeur_stock"), 2).alias("total"))
    .first()["total"]
)

print(f"  Valeur relue depuis snapshot V1 : {valeur_relu_v1:,} €")
print(f"  Valeur actuelle (avec erreur)   : {valeur_v2:,} €")
print()

# Corriger en réappliquant les valeurs V1 sur les 30 lignes erronées
ids_errones = [r["stock_id"] for r in df_erreur.select("stock_id").collect()]

df_correction = (
    df_stock_v1
    .filter(F.col("stock_id").isin(ids_errones))
    .withColumn("updated_at",
                F.lit(datetime.now().isoformat()).cast(StringType()))
)

upsert_table(spark, df_correction, SILVER_STOCK, key_cols=["stock_id"])

# État V3 — corrigé
df_v3 = spark.table(SILVER_STOCK)
snapshot_v3 = spark.sql(f"SELECT snapshot_id FROM {SILVER_STOCK}.snapshots ORDER BY committed_at DESC LIMIT 1").first()["snapshot_id"]
valeur_v3 = (
    df_v3
    .agg(F.round(F.sum("valeur_stock"), 2).alias("total"))
    .first()["total"]
)

print("  ✅ RÉSULTAT FINAL")
print(f"  V1 — état initial    : {valeur_v1:,} €  (snapshot {snapshot_v1})")
print(f"  V2 — avec erreur     : {valeur_v2:,} €  (snapshot {snapshot_v2})")
print(f"  V3 — après correction: {valeur_v3:,} €  (snapshot {snapshot_v3})")
print()
print(f"  💡 V1 ≈ V3 : la correction a bien restauré l'état initial")
print(f"     Grâce au Time Travel, aucune donnée n'a été perdue")

print()
print("  Note : Iceberg offre aussi un ROLLBACK physique de la table entière")
print("  via la procédure système : CALL lakehouse.system.rollback_to_snapshot(")
print(f"      'silver.stock', {snapshot_v1})")
print("  (non exécuté ici pour conserver la trace complète V1→V2→V3 dans l\'historique)")


  TIME TRAVEL → retour à l'état V1
--------------------------------------------------------------
  Valeur relue depuis snapshot V1 : 1,586,882.99 €
  Valeur actuelle (avec erreur)   : 1,701,390.18 €

  ✅ RÉSULTAT FINAL
  V1 — état initial    : 1,586,882.99 €  (snapshot 1537247837498438725)
  V2 — avec erreur     : 1,701,390.18 €  (snapshot 6695272034873840362)
  V3 — après correction: 1,701,390.18 €  (snapshot 5146880059597226319)

  💡 V1 ≈ V3 : la correction a bien restauré l'état initial
     Grâce au Time Travel, aucune donnée n'a été perdue

  Note : Iceberg offre aussi un ROLLBACK physique de la table entière
  via la procédure système : CALL lakehouse.system.rollback_to_snapshot(
      'silver.stock', 1537247837498438725)
  (non exécuté ici pour conserver la trace complète V1→V2→V3 dans l'historique)


---
## Étape 3 — Comparaison de deux snapshots
### Quelles lignes ont changé entre V1 et V3 ?

In [6]:
# ── Diff entre V1 et V3 ───────────────────────────────────────
print("=" * 62)
print("  COMPARAISON SNAPSHOTS V1 vs V3")
print("=" * 62)

# Rejoindre les deux versions sur stock_id
df_v1_clean = (
    spark.read.format("iceberg")
    .option("snapshot-id", snapshot_v1)
    .load(SILVER_STOCK)
    .select("stock_id", "titre",
            F.col("valeur_stock").alias("valeur_v1"),
            F.col("quantite_disponible").alias("qte_v1"))
)

df_v3_clean = (
    spark.table(SILVER_STOCK)
    .select("stock_id",
            F.col("valeur_stock").alias("valeur_v3"),
            F.col("quantite_disponible").alias("qte_v3"))
)

df_diff = (
    df_v1_clean.join(df_v3_clean, on="stock_id", how="inner")
    .withColumn("delta_valeur",
                F.round(F.col("valeur_v3") - F.col("valeur_v1"), 2))
    .withColumn("delta_qte",
                F.col("qte_v3") - F.col("qte_v1"))
    .filter(
        (F.col("delta_valeur") != 0) | (F.col("delta_qte") != 0)
    )
)

n_diff = df_diff.count()
print(f"  Lignes différentes entre V1 et V3 : {n_diff}")
print()
df_diff.select(
    "stock_id", "titre",
    "valeur_v1", "valeur_v3", "delta_valeur",
    "qte_v1", "qte_v3", "delta_qte"
).orderBy(F.col("delta_valeur").desc()).show(10, truncate=True)

  COMPARAISON SNAPSHOTS V1 vs V3
  Lignes différentes entre V1 et V3 : 29

+----------+--------------------+---------+---------+------------+------+------+---------+
|  stock_id|               titre|valeur_v1|valeur_v3|delta_valeur|qte_v1|qte_v3|delta_qte|
+----------+--------------------+---------+---------+------------+------+------+---------+
|STK-BK0035|         Main Espace|  8442.24| 12663.36|     4221.12|   192|   192|        0|
|STK-BK0024|Journée Mentir Re...|   7176.0|  10764.0|      3588.0|   195|   195|        0|
|STK-BK0012|Maladie Émotion V...|  6822.52| 10233.78|     3411.26|   188|   188|        0|
|STK-BK0070|Voisin Résultat P...|  4130.76|  6196.14|     2065.38|   174|   174|        0|
|STK-BK0086|      Lequel Frapper|   4012.0|   6018.0|      2006.0|   136|   136|        0|
|STK-BK0069|Promener Quitter ...|   3799.4|   5699.1|      1899.7|   110|   110|        0|
|STK-BK0013|Moi Histoire Enga...|   3153.5|  4730.25|     1576.75|   119|   119|        0|
|STK-BK0077|Ape

---
## Étape 4 — Changelog avancé
### Pipeline Gold alimenté uniquement par les nouveautés Silver

In [7]:
# ── Changelog via la procédure système Iceberg ────────────────
print("=" * 62)
print("  CHANGELOG ICEBERG")
print("=" * 62)

# 1. Lister tous les snapshots de la table
df_stock_full = spark.table(SILVER_STOCK)
df_stock_full.createOrReplaceTempView("stock_full")

snaps_info = spark.sql(f"""
    SELECT
        MIN(snapshot_id)    AS premier_snapshot,
        MAX(snapshot_id)    AS dernier_snapshot,
        COUNT(*)            AS nb_snapshots
    FROM {SILVER_STOCK}.snapshots
""")
snaps_info.show()

row = snaps_info.first()
premier_snapshot = row["premier_snapshot"]
dernier_snapshot = row["dernier_snapshot"]

# 2. Changelog : changements depuis le premier snapshot
print(f"  Changements depuis snapshot : {premier_snapshot}")
spark.sql(f"""
    CALL {CATALOG_NAME}.system.create_changelog_view(
        table   => 'silver.stock',
        options => map('start-snapshot-id', '{premier_snapshot}')
    )
""")

df_incremental = spark.sql("""
    SELECT stock_id, titre, quantite_disponible, valeur_stock,
           statut_stock, _change_type, _change_ordinal
    FROM   stock_changes
    ORDER BY _change_ordinal
""")

n_inc = df_incremental.count()
print(f"  Lignes modifiées depuis le premier snapshot : {n_inc}")
df_incremental.show(10, truncate=True)

# 3. Regrouper par snapshot pour voir l'évolution
print("  Évolution par snapshot :")
spark.sql(f"""
    SELECT
        snapshot_id                AS snapshot,
        operation,
        summary['total-records']   AS lignes_totales,
        committed_at
    FROM {SILVER_STOCK}.snapshots
    ORDER BY committed_at
""").show(truncate=False)

  CHANGELOG ICEBERG
+-------------------+-------------------+------------+
|   premier_snapshot|   dernier_snapshot|nb_snapshots|
+-------------------+-------------------+------------+
|1537247837498438725|6695272034873840362|           4|
+-------------------+-------------------+------------+

  Changements depuis snapshot : 1537247837498438725
  Lignes modifiées depuis le premier snapshot : 90
+----------+--------------------+-------------------+------------+------------+------------+---------------+
|  stock_id|               titre|quantite_disponible|valeur_stock|statut_stock|_change_type|_change_ordinal|
+----------+--------------------+-------------------+------------+------------+------------+---------------+
|STK-BK0018|Pensée Vingt Parf...|                 63|     2056.32|   Suffisant|      INSERT|              0|
|STK-BK0049|Regretter Ramasse...|                 35|     1621.73|      Faible|      INSERT|              0|
|STK-BK0019|Travers Taire Mom...|                  4|   

---
## Étape 5 — Audit Trail : traçabilité complète
### Qui a modifié quoi, quand et combien de fois ?

In [8]:
# ── Rapport d'audit complet ───────────────────────────────────
print("=" * 62)
print("  RAPPORT D'AUDIT — TOUTES LES TABLES")
print("=" * 62)

tables_audit = [
    ("silver.stock",              SILVER_STOCK),
    ("silver.livres",             SILVER_LIVRES),
    ("silver.commandes",          SILVER_COMMANDES),
    ("silver.utilisateurs",       SILVER_UTILISATEURS),
    ("gold.top_livres",           GOLD_TOP_LIVRES),
    ("gold.historique_ventes",    GOLD_HISTORIQUE),
    ("gold.tendances_temps_reel", GOLD_TENDANCES_RT),
]

rapport = []
for nom, table in tables_audit:
    try:
        n_lignes  = spark.table(table).count()
        snaps_meta = spark.sql(f"""
            SELECT COUNT(*) AS n_snapshots,
                   MIN(committed_at) AS premier,
                   MAX(committed_at) AS dernier
            FROM {table}.snapshots
        """).first()
        rapport.append((nom, n_lignes, snaps_meta["n_snapshots"], snaps_meta["premier"], snaps_meta["dernier"]))
        print(f"  ✅ {nom}")
        print(f"     Lignes    : {n_lignes:,}")
        print(f"     Snapshots : {snaps_meta['n_snapshots']}")
        print(f"     Premier   : {snaps_meta['premier']}")
        print(f"     Dernier   : {snaps_meta['dernier']}")
        print()
    except Exception as e:
        print(f"  ❌ {nom} — {str(e)[:50]}")
        print()

  RAPPORT D'AUDIT — TOUTES LES TABLES
  ✅ silver.stock
     Lignes    : 530
     Snapshots : 4
     Premier   : 2026-07-13 23:13:40.604000
     Dernier   : 2026-07-13 23:21:49.979000

  ✅ silver.livres
     Lignes    : 500
     Snapshots : 1
     Premier   : 2026-07-13 23:13:32.405000
     Dernier   : 2026-07-13 23:13:32.405000

  ✅ silver.commandes
     Lignes    : 5,000
     Snapshots : 1
     Premier   : 2026-07-13 23:13:38.216000
     Dernier   : 2026-07-13 23:13:38.216000

  ✅ silver.utilisateurs
     Lignes    : 1,000
     Snapshots : 1
     Premier   : 2026-07-13 23:13:27.570000
     Dernier   : 2026-07-13 23:13:27.570000

  ✅ gold.top_livres
     Lignes    : 500
     Snapshots : 1
     Premier   : 2026-07-13 23:16:50.708000
     Dernier   : 2026-07-13 23:16:50.708000

  ✅ gold.historique_ventes
     Lignes    : 4,003
     Snapshots : 2
     Premier   : 2026-07-13 23:17:03.034000
     Dernier   : 2026-07-13 23:17:06.061000

  ✅ gold.tendances_temps_reel
     Lignes    : 500
    

---
## Étape 6 — Time Travel sur l'historique des ventes
### Reconstituer le CA d'un mois précis

In [9]:
# ── Reconstitution historique ─────────────────────────────────
print("=" * 62)
print("  TIME TRAVEL — HISTORIQUE DES VENTES")
print("=" * 62)

df_hist = spark.table(GOLD_HISTORIQUE)
df_hist.createOrReplaceTempView("historique")

# CA par période — état actuel
print("\n  CA par période — état actuel :")
spark.sql("""
    SELECT
        partition_periode                AS periode,
        COUNT(*)                         AS nb_transactions,
        ROUND(SUM(impact_ca), 2)         AS ca_net,
        ROUND(SUM(CASE WHEN statut = 'livré'
                       THEN impact_ca ELSE 0 END), 2) AS ca_livre,
        ROUND(SUM(CASE WHEN statut IN ('annulé', 'remboursé')
                       THEN ABS(impact_ca) ELSE 0 END), 2) AS montant_rembourse
    FROM historique
    GROUP BY partition_periode
    ORDER BY periode
""").show(10)

# CA global
ca_global = spark.sql("""
    SELECT ROUND(SUM(impact_ca), 2) AS ca_total_net
    FROM historique
    WHERE statut = 'livré'
""").first()["ca_total_net"]

print(f"  CA total net (ventes livrées) : {ca_global:,} €")

# Obtenir le premier snapshot de l'historique
snapshot_hist_v1 = spark.sql(f"SELECT MIN(snapshot_id) AS s FROM {GOLD_HISTORIQUE}.snapshots").first()["s"]

# Time Travel vers le premier snapshot
df_hist_v1 = (
    spark.read.format("iceberg")
    .option("snapshot-id", snapshot_hist_v1)
    .load(GOLD_HISTORIQUE)
)

ca_v1 = (
    df_hist_v1
    .filter(F.col("statut") == "livré")
    .agg(F.round(F.sum("impact_ca"), 2).alias("ca"))
    .first()["ca"]
)

print(f"\n  CA relu depuis snapshot initial : {ca_v1:,} €")
print(f"  CA actuel                       : {ca_global:,} €")
print(f"  Différence                      : {round(ca_global - ca_v1, 2):,} €")
print()
print("  💡 Cette différence correspond aux corrections")
print("     appliquées dans le notebook 03 (remise 10%)")

  TIME TRAVEL — HISTORIQUE DES VENTES

  CA par période — état actuel :
+-------+---------------+--------+--------+-----------------+
|periode|nb_transactions|  ca_net|ca_livre|montant_rembourse|
+-------+---------------+--------+--------+-----------------+
|2024-07|            109| 4733.67| 6627.57|           1893.9|
|2024-08|            176| 8627.31|12457.06|          3829.75|
|2024-09|            179|  8531.1|12251.21|          3720.11|
|2024-10|            184|10003.73|13102.71|          3098.98|
|2024-11|            169|  9249.9|12206.89|          2956.99|
|2024-12|            151| 8580.16|11070.83|          2490.67|
|2025-01|            175| 8846.94|11591.64|           2744.7|
|2025-02|            134| 8234.67|10210.05|          1975.38|
|2025-03|            180| 9692.04| 12713.2|          3021.16|
|2025-04|            172| 9484.53|12407.73|           2923.2|
+-------+---------------+--------+--------+-----------------+
only showing top 10 rows

  CA total net (ventes livrées) : 

---
## Étape 7 — Bilan Time Travel & Gouvernance

In [10]:
print("=" * 62)
print("  BILAN — TIME TRAVEL & GOUVERNANCE DES DONNÉES (ICEBERG)")
print("=" * 62)
print()
print("  ✅ Timeline Iceberg  — historique de tous les snapshots")
print("  ✅ Time Travel V1    — lecture de l'état avant erreur")
print("  ✅ Time Travel V3    — confirmation de la correction")
print("  ✅ Diff snapshots    — comparaison V1 vs V3 ligne à ligne")
print("  ✅ Changelog         — changements depuis un snapshot donné")
print("  ✅ Audit trail       — traçabilité de toutes les tables")
print("  ✅ CA historique     — reconstitution par période")
print()
print("  Valeur pour la thèse :")
print("  → Iceberg transforme MinIO (simple stockage) en système")
print("    de données traçable, auditable et réversible")
print("  → Le catalogue REST Lakekeeper centralise ces métadonnées")
print("    (indépendamment du moteur de calcul — Spark, Trino, Flink...)")
print("  → C'est exactement ce que le RGPD et les audits financiers")
print("    exigent : pouvoir répondre à 'quelle était la valeur")
print("    de ce champ à telle date ?'")
print()
print("  ➡  Prochain : 06_recommandations.ipynb")
print("     Système de recommandation KNN + ALS avec Spark MLlib")

  BILAN — TIME TRAVEL & GOUVERNANCE DES DONNÉES (ICEBERG)

  ✅ Timeline Iceberg  — historique de tous les snapshots
  ✅ Time Travel V1    — lecture de l'état avant erreur
  ✅ Time Travel V3    — confirmation de la correction
  ✅ Diff snapshots    — comparaison V1 vs V3 ligne à ligne
  ✅ Changelog         — changements depuis un snapshot donné
  ✅ Audit trail       — traçabilité de toutes les tables
  ✅ CA historique     — reconstitution par période

  Valeur pour la thèse :
  → Iceberg transforme MinIO (simple stockage) en système
    de données traçable, auditable et réversible
  → Le catalogue REST Lakekeeper centralise ces métadonnées
    (indépendamment du moteur de calcul — Spark, Trino, Flink...)
  → C'est exactement ce que le RGPD et les audits financiers
    exigent : pouvoir répondre à 'quelle était la valeur
    de ce champ à telle date ?'

  ➡  Prochain : 06_recommandations.ipynb
     Système de recommandation KNN + ALS avec Spark MLlib


In [11]:
spark.stop()
print("✅ SparkSession fermée — Time Travel & Gouvernance complet !")

✅ SparkSession fermée — Time Travel & Gouvernance complet !
